# Lab 11 — DATS 6103

**Name:**

This notebook mirrors the lab page. Everything here also runs in the browser at
<https://www.smajhi.com/DATS-6103/labs/knn.html>, where the self-checks and the timers live; work in whichever you
prefer. The discussion half of the lab is not in this file — it happens away
from the keyboard.

**Conditions.** No AI assistants. The official Python, NumPy, Pandas,
Matplotlib and scikit-learn documentation and the course notes may be consulted
freely.

Run the self-check under each answer. It tells you whether your answer is right
without telling you what the answer is.

## Part A—Work it out

**Not submitted, and not written at the keyboard.** Work these in your team, on paper, one problem at a time; each is reported to the class as you finish it. The hint cards are on the lab page: <https://www.smajhi.com/DATS-6103/labs/knn.html>.

### A1. Dollars against years

*Suggested: 7 minutes.*

A store predicts whether a customer buys, from **annual income** in dollars and
**age** in years. A new customer earns \$50,000 and is 30. Four past customers:

| | Income (\$) | Age | Bought? |
|:--|--:|--:|:--|
| P1 | 51,000 | 60 | yes |
| P2 | 55,000 | 31 | no |
| P3 | 49,500 | 45 | yes |
| P4 | 58,000 | 29 | no |

a. Compute the Euclidean distance from the new customer to each, in the raw
   units. What do 1-NN and 3-NN predict?
b. Now standardize first. In the training data the standard deviation of income
   is \$10,000 and of age is 10 years. Recompute the four distances and both
   predictions.
c. Which pair of predictions is right?

### A2. Everyone is equally far away

*Suggested: 8 minutes.*

Draw two points at random in the unit cube $[0,1]^d$, each coordinate independent
and uniform. For one coordinate, $E[(X - Y)^2] = 1/6$ and
$\operatorname{Var}[(X - Y)^2] = 7/180$.

a. What are the mean and the standard deviation of the **squared** distance
   between the two points, in $d$ dimensions?
b. Their ratio, the standard deviation over the mean, measures how much
   distances differ from one pair to the next. Compute it for $d = 1$, $100$ and
   $10{,}000$.
c. What does that do to "the nearest neighbor"? And what is the reading's
   advice for it?

### A3. An ROC curve by hand

*Suggested: 10 minutes.*

A 5-NN model scores 20 test tumors, 10 malignant and 10 benign. A tumor's score
is the fraction of its five neighbors that are malignant, so it is always 0, 0.2,
0.4, 0.6, 0.8 or 1. The rule calls a tumor malignant when its score is at least
$\tau$; malignant is the positive class. Five thresholds give five confusion
matrices:

| $\tau$ | TP | FN | FP | TN |
|--:|--:|--:|--:|--:|
| 0.2 | 10 | 0 | 6 | 4 |
| 0.4 | 9 | 1 | 4 | 6 |
| 0.6 | 8 | 2 | 2 | 8 |
| 0.8 | 7 | 3 | 1 | 9 |
| 1.0 | 4 | 6 | 0 | 10 |

a. Compute the true positive rate and the false positive rate at each
   threshold. Then add the two thresholds the table leaves out, $\tau = 0$ and
   any $\tau$ above 1. Why would $\tau = 0.5$ add nothing new?
b. Draw the ROC curve: plot the seven points with the false positive rate on the
   horizontal axis and join neighbors with straight lines. Compute the area
   under it with trapezoids.
c. A missed malignancy costs far more than a false alarm. Which threshold do you
   choose, and what does it cost? What area would a model that scores at random
   have?

## Part B—Build

### B1. Choosing k

*Suggested: 12 minutes.*

The breast cancer data from Lab 10's final: 569 tumors, 30 measurements,
malignant or benign. For each $k$ in `ks`, record the **5-fold cross-validated**
accuracy of `knn(k)` on the training data, using `folds`. Then refit the best
$k$ and score it on the test set.

- `cv_acc`---a list, one entry per $k$;
- `best_k`---the $k$ with the highest cross-validated accuracy;
- `train_acc_1`---the accuracy of `knn(1)` on the **training** data it was fit
  on;
- `test_acc`---`knn(best_k)`, fit on all of `X_train`, scored on `X_test`.

**Already loaded:** `bc`, `X_train`, `X_test`, `y_train` and `y_test`, `folds`,
`ks`, and `knn(k)`, which returns a new, unfitted pipeline: a `StandardScaler`
followed by a k-NN classifier with that $k$.

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
bc = load_breast_cancer()
X, y = bc.data, bc.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=6103, stratify=y)
folds = StratifiedKFold(5, shuffle=True, random_state=6103)
ks = [1, 3, 5, 7, 9, 11, 15, 21, 31]
def knn(k):
    return make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
def show(v):
    """How a value prints: a blank as 'blank', a number to three decimals."""
    if v is ...:
        return "blank"
    if isinstance(v, (float, np.floating)):
        return f"{v:.3f}"
    if isinstance(v, dict):
        return "{" + ", ".join(f"{k}: {show(x)}" for k, x in v.items()) + "}"
    if isinstance(v, (list, tuple, np.ndarray)):
        return "[" + ", ".join(show(x) for x in v) + "]"
    return str(v)

In [ ]:
cv_acc = []
for k in ks:
    cv_acc.append(...)
best_k = ...
train_acc_1 = ...
test_acc = ...

print(f"{'k':>4} {'cv':>7}")
for k, c in zip(ks, cv_acc):
    print(f"{k:>4} {show(c):>7}")
print("best k:", show(best_k), "| 1-NN on its own training data:", show(train_acc_1),
      "| test accuracy:", show(test_acc))

In [ ]:
# self-check: run this after your answer above
wc = [cross_val_score(knn(k_), X_train, y_train, cv=folds).mean() for k_ in ks]
wb = ks[int(np.argmax(wc))]
wt1 = knn(1).fit(X_train, y_train).score(X_train, y_train)
wtest = knn(wb).fit(X_train, y_train).score(X_test, y_test)
def _blank(v):
    return (v is ... or (isinstance(v, (list, tuple)) and any(_blank(x) for x in v))
            or (isinstance(v, dict) and any(_blank(x) for x in v.values())))
def _part(name, value, test, hint):
    if _blank(value):
        return False, f"{name}: not filled in yet."
    try:
        good = bool(test())
    except Exception:
        good = False
    return good, (f"{name}: right." if good else f"{name}: {hint}")
globals().update(locals())                 # the page runs checks with separate globals and locals
_parts = [
    _part("cv_acc", cv_acc, lambda: np.allclose(cv_acc, wc),
          "should be cross_val_score(knn(k), X_train, y_train, cv=folds).mean(), one entry per k."),
    _part("best_k", best_k, lambda: best_k == wb,
          "should maximize the cross-validated accuracy."),
    _part("train_acc_1", train_acc_1, lambda: abs(float(train_acc_1) - wt1) < 1e-9,
          "should be knn(1) fit on X_train and scored on X_train."),
    _part("test_acc", test_acc, lambda: abs(float(test_acc) - wtest) < 1e-9,
          "should be knn(best_k), fit on all of X_train and scored on X_test."),
]
assert all(g for g, _ in _parts), " ".join(m for _, m in _parts)
print("looks right")

### B2. What the units cost

*Suggested: 12 minutes.*

A1 on real data. Compare k-NN with `k = best_k` **with and without** the scaler,
by cross-validated accuracy, and find the column that dominates the raw
distances.

- `cv_raw`---5-fold cross-validated accuracy of a `KNeighborsClassifier` with
  `best_k` neighbors, **no scaler**;
- `cv_scaled`---the same with `knn(best_k)`;
- `top_feature`---the name, from `bc.feature_names`, of the training column with
  the largest variance;
- `top_share`---its variance as a share of the total variance of all 30
  training columns.

**Already loaded:** everything from B1, and `best_k`, set to B1's answer.

In [ ]:
# uses best_k from B1 above: run B1 first

In [ ]:
cv_raw = ...
cv_scaled = ...
variances = ...
top_feature = ...
top_share = ...

print("raw:", show(cv_raw), "| scaled:", show(cv_scaled))
print("largest-variance column:", show(top_feature), "| share of total variance:", show(top_share))

In [ ]:
# self-check: run this after your answer above
wr = cross_val_score(KNeighborsClassifier(n_neighbors=best_k), X_train, y_train, cv=folds).mean()
ws = cross_val_score(knn(best_k), X_train, y_train, cv=folds).mean()
v_ = X_train.var(axis=0)
wf = str(bc.feature_names[int(np.argmax(v_))])
wsh = v_.max() / v_.sum()
def _blank(v):
    return (v is ... or (isinstance(v, (list, tuple)) and any(_blank(x) for x in v))
            or (isinstance(v, dict) and any(_blank(x) for x in v.values())))
def _part(name, value, test, hint):
    if _blank(value):
        return False, f"{name}: not filled in yet."
    try:
        good = bool(test())
    except Exception:
        good = False
    return good, (f"{name}: right." if good else f"{name}: {hint}")
globals().update(locals())                 # the page runs checks with separate globals and locals
_parts = [
    _part("cv_raw", cv_raw, lambda: abs(float(cv_raw) - wr) < 1e-9,
          "should be KNeighborsClassifier(n_neighbors=best_k) with no scaler, cross-validated on X_train with folds."),
    _part("cv_scaled", cv_scaled, lambda: abs(float(cv_scaled) - ws) < 1e-9,
          "should be knn(best_k), cross-validated on X_train with folds."),
    _part("top_feature", top_feature, lambda: str(top_feature) == wf,
          "should be the name in bc.feature_names of the column of X_train with the largest variance."),
    _part("top_share", top_share, lambda: abs(float(top_share) - wsh) < 1e-6,
          "should be that column's variance divided by the sum of all 30 variances, on X_train."),
]
assert all(g for g, _ in _parts), " ".join(m for _, m in _parts)
print("looks right")

### B3. Thirty columns of noise

*Suggested: 14 minutes.*

A2 on real data. `with_noise(X, m)` returns `X` with `m` extra columns of
standard normal noise appended, the same noise every time for the same `m`.
For each `m` in `ms`, record the 5-fold cross-validated accuracy, on
`with_noise(X_train, m)`, of `knn(best_k)` and of `logit()`, a scaled logistic
regression.

- `knn_acc`, `logit_acc`---lists, one entry per `m`;
- `knn_drop`, `logit_drop`---each model's accuracy with no noise **minus** its
  accuracy at the largest `m`.

**Already loaded:** everything from B2, `ms`, `with_noise(X, m)` and `logit()`.

In [ ]:
# uses best_k from B1 above: run B1 first

from sklearn.linear_model import LogisticRegression
ms = [0, 10, 30, 100, 300]
def logit():
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
def with_noise(X, m):
    """X with m columns of standard normal noise appended; the same noise for the same m."""
    rng = np.random.default_rng(0)
    return np.hstack([X, rng.standard_normal((len(X), m))])

In [ ]:
knn_acc, logit_acc = [], []
for m in ms:
    Xm = with_noise(X_train, m)
    knn_acc.append(...)
    logit_acc.append(...)
knn_drop = ...
logit_drop = ...

print(f"{'noise':>6} {'k-NN':>7} {'logistic':>9}")
for m, a, b in zip(ms, knn_acc, logit_acc):
    print(f"{m:>6} {show(a):>7} {show(b):>9}")
print("drop, k-NN:", show(knn_drop), "| drop, logistic:", show(logit_drop))

In [ ]:
# self-check: run this after your answer above
wk = [cross_val_score(knn(best_k), with_noise(X_train, m_), y_train, cv=folds).mean() for m_ in ms]
wl = [cross_val_score(logit(), with_noise(X_train, m_), y_train, cv=folds).mean() for m_ in ms]
def _blank(v):
    return (v is ... or (isinstance(v, (list, tuple)) and any(_blank(x) for x in v))
            or (isinstance(v, dict) and any(_blank(x) for x in v.values())))
def _part(name, value, test, hint):
    if _blank(value):
        return False, f"{name}: not filled in yet."
    try:
        good = bool(test())
    except Exception:
        good = False
    return good, (f"{name}: right." if good else f"{name}: {hint}")
globals().update(locals())                 # the page runs checks with separate globals and locals
_parts = [
    _part("knn_acc", knn_acc, lambda: np.allclose(knn_acc, wk),
          "should be cross_val_score(knn(best_k), Xm, y_train, cv=folds).mean(), one entry per m."),
    _part("logit_acc", logit_acc, lambda: np.allclose(logit_acc, wl),
          "should be cross_val_score(logit(), Xm, y_train, cv=folds).mean(), one entry per m."),
    _part("knn_drop", knn_drop, lambda: abs(float(knn_drop) - (wk[0] - wk[-1])) < 1e-9,
          "should be k-NN's accuracy at m = 0 minus its accuracy at the largest m."),
    _part("logit_drop", logit_drop, lambda: abs(float(logit_drop) - (wl[0] - wl[-1])) < 1e-9,
          "should be the logistic regression's accuracy at m = 0 minus its accuracy at the largest m."),
]
assert all(g for g, _ in _parts), " ".join(m for _, m in _parts)
print("looks right")

## Before you leave

Upload this notebook to Blackboard. It is the record of what you did; the lab
grade itself comes from what you do in the room.

Then post a comment at the bottom of the lab page, <https://www.smajhi.com/DATS-6103/labs/knn.html>. One line is enough.
It proves your GitHub account works with the comment system, which is how every
week's reading credit is earned.